In [ ]:
# ==============================================================
#              AUTOMATED FILE ORGANIZER
#                   INTERMEDIATE PROJECT
# ==============================================================
# Language : Python
# GUI      : Tkinter
# Purpose  : Automatically organize files into folders
#            according to their file extensions.
#
# This program allows the user to:
# 1. Select a folder
# 2. Scan files in that folder
# 3. Create category folders automatically
# 4. Move files according to their file type
# 5. Handle duplicate file names safely
# 6. Display an organization summary
# ==============================================================


# --------------------------------------------------------------
# 1. IMPORT REQUIRED LIBRARIES
# --------------------------------------------------------------

import os
import shutil
import tkinter as tk

from tkinter import ttk
from tkinter import filedialog
from tkinter import messagebox


# --------------------------------------------------------------
# 2. FILE CATEGORY DEFINITIONS
# --------------------------------------------------------------
# Each category contains common file extensions.
# The program uses these extensions to decide where a file
# should be moved.

FILE_CATEGORIES = {

    "Documents": [
        ".pdf",
        ".doc",
        ".docx",
        ".txt",
        ".xls",
        ".xlsx",
        ".ppt",
        ".pptx",
        ".csv"
    ],

    "Images": [
        ".jpg",
        ".jpeg",
        ".png",
        ".gif",
        ".bmp",
        ".webp",
        ".svg",
        ".tiff"
    ],

    "Videos": [
        ".mp4",
        ".mkv",
        ".avi",
        ".mov",
        ".wmv",
        ".flv",
        ".webm"
    ],

    "Audio": [
        ".mp3",
        ".wav",
        ".aac",
        ".ogg",
        ".flac",
        ".m4a"
    ],

    "Archives": [
        ".zip",
        ".rar",
        ".7z",
        ".tar",
        ".gz"
    ],

    "Programs": [
        ".py",
        ".java",
        ".c",
        ".cpp",
        ".html",
        ".css",
        ".js",
        ".json"
    ],

    "Installers": [
        ".exe",
        ".msi",
        ".apk"
    ]
}


# --------------------------------------------------------------
# 3. CREATE MAIN APPLICATION WINDOW
# --------------------------------------------------------------

root = tk.Tk()

# Set the title of the application.
root.title("Automated File Organizer")

# Set the window size.
root.geometry("900x650")

# Prevent resizing.
root.resizable(False, False)


# --------------------------------------------------------------
# 4. APPLICATION VARIABLES
# --------------------------------------------------------------

# Stores the folder selected by the user.
folder_var = tk.StringVar()

# Stores the organization status.
status_var = tk.StringVar(
    value="Select a folder to begin."
)

# Stores the number of files processed.
total_var = tk.StringVar(
    value="Total Files: 0"
)

# Stores the number of files organized.
organized_var = tk.StringVar(
    value="Organized: 0"
)


# --------------------------------------------------------------
# 5. APPLICATION TITLE
# --------------------------------------------------------------

title_label = tk.Label(
    root,
    text="AUTOMATED FILE ORGANIZER",
    font=("Arial", 24, "bold")
)

title_label.pack(pady=20)


# --------------------------------------------------------------
# 6. DESCRIPTION
# --------------------------------------------------------------

description_label = tk.Label(
    root,
    text=(
        "Select a folder and automatically organize files "
        "into categorized folders."
    ),
    font=("Arial", 11)
)

description_label.pack(pady=5)


# --------------------------------------------------------------
# 7. FOLDER SELECTION FRAME
# --------------------------------------------------------------

folder_frame = tk.Frame(root)

folder_frame.pack(
    pady=20
)


# Folder label
tk.Label(
    folder_frame,
    text="Selected Folder:",
    font=("Arial", 11, "bold")
).grid(
    row=0,
    column=0,
    padx=5,
    pady=5
)


# Entry displaying the selected folder.
folder_entry = tk.Entry(
    folder_frame,
    textvariable=folder_var,
    width=60
)

folder_entry.grid(
    row=0,
    column=1,
    padx=5,
    pady=5
)


# --------------------------------------------------------------
# 8. SELECT FOLDER FUNCTION
# --------------------------------------------------------------

def select_folder():

    # Open Windows folder selection dialog.
    selected_folder = filedialog.askdirectory(
        title="Select Folder to Organize"
    )

    # Update the entry when a folder is selected.
    if selected_folder:

        folder_var.set(
            selected_folder
        )

        status_var.set(
            "Folder selected. Ready to organize."
        )


# --------------------------------------------------------------
# 9. BROWSE BUTTON
# --------------------------------------------------------------

browse_button = tk.Button(
    folder_frame,
    text="Browse",
    command=select_folder,
    width=12
)

browse_button.grid(
    row=0,
    column=2,
    padx=5
)


# --------------------------------------------------------------
# 10. SUMMARY FRAME
# --------------------------------------------------------------

summary_frame = tk.LabelFrame(
    root,
    text="Organization Summary",
    padx=20,
    pady=15
)

summary_frame.pack(
    padx=30,
    pady=10,
    fill="x"
)


# Total files label
tk.Label(
    summary_frame,
    textvariable=total_var,
    font=("Arial", 11, "bold")
).grid(
    row=0,
    column=0,
    padx=30,
    pady=10
)


# Organized files label
tk.Label(
    summary_frame,
    textvariable=organized_var,
    font=("Arial", 11, "bold")
).grid(
    row=0,
    column=1,
    padx=30,
    pady=10
)


# --------------------------------------------------------------
# 11. OUTPUT TABLE
# --------------------------------------------------------------

table_frame = tk.Frame(root)

table_frame.pack(
    padx=30,
    pady=10
)


# Define columns for the table.
columns = (
    "File Name",
    "Extension",
    "Category",
    "Status"
)


# Create table.
file_tree = ttk.Treeview(
    table_frame,
    columns=columns,
    show="headings",
    height=12
)


# Create column headings.
for column in columns:

    file_tree.heading(
        column,
        text=column
    )


# Set column sizes.
file_tree.column(
    "File Name",
    width=300
)

file_tree.column(
    "Extension",
    width=100
)

file_tree.column(
    "Category",
    width=150
)

file_tree.column(
    "Status",
    width=150
)


# Create vertical scrollbar.
scrollbar = ttk.Scrollbar(
    table_frame,
    orient="vertical",
    command=file_tree.yview
)


# Connect scrollbar to table.
file_tree.configure(
    yscrollcommand=scrollbar.set
)


# Display table.
file_tree.pack(
    side="left"
)

# Display scrollbar.
scrollbar.pack(
    side="right",
    fill="y"
)


# --------------------------------------------------------------
# 12. DETERMINE FILE CATEGORY
# --------------------------------------------------------------

def get_category(file_name):

    # Extract the file extension.
    extension = os.path.splitext(
        file_name
    )[1].lower()

    # Check each category.
    for category, extensions in FILE_CATEGORIES.items():

        if extension in extensions:

            return category

    # If no category matches, put it into Others.
    return "Others"


# --------------------------------------------------------------
# 13. GENERATE UNIQUE FILE NAME
# --------------------------------------------------------------

def get_unique_path(destination_folder, file_name):

    # Build the original destination path.
    original_path = os.path.join(
        destination_folder,
        file_name
    )

    # If the file does not already exist,
    # use the original name.
    if not os.path.exists(original_path):

        return original_path

    # Separate filename and extension.
    base_name, extension = os.path.splitext(
        file_name
    )

    # Start numbering duplicate files.
    counter = 1

    # Continue until a free name is found.
    while True:

        new_name = (
            f"{base_name}_{counter}"
            f"{extension}"
        )

        new_path = os.path.join(
            destination_folder,
            new_name
        )

        if not os.path.exists(new_path):

            return new_path

        counter += 1


# --------------------------------------------------------------
# 14. ORGANIZE FILES FUNCTION
# --------------------------------------------------------------

def organize_files():

    # Get the selected folder.
    selected_folder = folder_var.get().strip()

    # Check whether a folder was selected.
    if selected_folder == "":

        messagebox.showwarning(
            "Folder Required",
            "Please select a folder first."
        )

        return

    # Verify the folder exists.
    if not os.path.isdir(
        selected_folder
    ):

        messagebox.showerror(
            "Invalid Folder",
            "The selected folder does not exist."
        )

        return

    # Clear the previous table contents.
    for item in file_tree.get_children():

        file_tree.delete(item)

    # Reset counters.
    total_files = 0
    organized_files = 0

    # Get all entries in the folder.
    try:

        entries = os.listdir(
            selected_folder
        )

    except Exception as error:

        messagebox.showerror(
            "Access Error",
            f"Unable to read the folder.\n\n{error}"
        )

        return

    # Process each item.
    for item in entries:

        # Create complete source path.
        source_path = os.path.join(
            selected_folder,
            item
        )

        # Ignore folders.
        if not os.path.isfile(
            source_path
        ):

            continue

        # Count the file.
        total_files += 1

        # Identify the file extension.
        extension = os.path.splitext(
            item
        )[1].lower()

        # Identify the category.
        category = get_category(
            item
        )

        # Create category folder path.
        destination_folder = os.path.join(
            selected_folder,
            category
        )

        # Create the category folder
        # if it does not already exist.
        try:

            os.makedirs(
                destination_folder,
                exist_ok=True
            )

        except Exception as error:

            file_tree.insert(
                "",
                tk.END,
                values=(
                    item,
                    extension,
                    category,
                    "Folder Error"
                )
            )

            continue

        # Generate a unique destination path.
        destination_path = get_unique_path(
            destination_folder,
            item
        )

        # Move the file.
        try:

            shutil.move(
                source_path,
                destination_path
            )

            # Increase organized file count.
            organized_files += 1

            # Add successful operation to table.
            file_tree.insert(
                "",
                tk.END,
                values=(
                    item,
                    extension if extension else "None",
                    category,
                    "Organized"
                )
            )

        except Exception as error:

            # Display failed operation.
            file_tree.insert(
                "",
                tk.END,
                values=(
                    item,
                    extension if extension else "None",
                    category,
                    "Failed"
                )
            )

    # Update summary.
    total_var.set(
        f"Total Files: {total_files}"
    )

    organized_var.set(
        f"Organized: {organized_files}"
    )

    # Update status.
    status_var.set(
        "File organization completed."
    )

    # Show completion message.
    messagebox.showinfo(
        "Completed",
        f"Organization completed successfully!\n\n"
        f"Total files found: {total_files}\n"
        f"Files organized: {organized_files}"
    )


# --------------------------------------------------------------
# 15. RESET APPLICATION FUNCTION
# --------------------------------------------------------------

def reset_application():

    # Clear selected folder.
    folder_var.set("")

    # Reset summary.
    total_var.set(
        "Total Files: 0"
    )

    organized_var.set(
        "Organized: 0"
    )

    # Reset status.
    status_var.set(
        "Select a folder to begin."
    )

    # Remove table contents.
    for item in file_tree.get_children():

        file_tree.delete(item)


# --------------------------------------------------------------
# 16. BUTTON FRAME
# --------------------------------------------------------------

button_frame = tk.Frame(root)

button_frame.pack(
    pady=15
)


# --------------------------------------------------------------
# 17. ORGANIZE BUTTON
# --------------------------------------------------------------

organize_button = tk.Button(
    button_frame,
    text="ORGANIZE FILES",
    command=organize_files,
    width=20,
    height=2,
    font=("Arial", 11, "bold")
)

organize_button.grid(
    row=0,
    column=0,
    padx=10
)


# --------------------------------------------------------------
# 18. RESET BUTTON
# --------------------------------------------------------------

reset_button = tk.Button(
    button_frame,
    text="RESET",
    command=reset_application,
    width=15,
    height=2,
    font=("Arial", 11, "bold")
)

reset_button.grid(
    row=0,
    column=1,
    padx=10
)


# --------------------------------------------------------------
# 19. STATUS LABEL
# --------------------------------------------------------------

status_label = tk.Label(
    root,
    textvariable=status_var,
    font=("Arial", 10, "italic")
)

status_label.pack(
    pady=10
)


# --------------------------------------------------------------
# 20. INFORMATION LABEL
# --------------------------------------------------------------

info_label = tk.Label(
    root,
    text=(
        "Files are categorized based on their extensions. "
        "Duplicate names are automatically renamed."
    ),
    font=("Arial", 9)
)

info_label.pack(
    pady=5
)


# --------------------------------------------------------------
# 21. START APPLICATION
# --------------------------------------------------------------

# Start the Tkinter event loop.
root.mainloop()


# ==============================================================
#                    END OF PROGRAM
# ==============================================================